# Induction Head Causal Tests

Goal: test whether the candidate induction heads identified earlier are causally important for next-token prediction on repeated-pattern prompts.

This notebook now covers the later experiment stages too: candidate selection among top detected heads, direct comparison of detection vs causal winners, and the multi-head ablation analysis that suggested a small cooperating head set.


## Setup

The benchmark logic lives in `scripts/benchmark.py`. The notebook stays readable by calling those helpers directly and then presenting the intermediate tables here.


In [ ]:
import json
import sys
from itertools import combinations
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.benchmark import (
    build_prompt_dataset,
    evaluate_head_set,
    load_model,
    run_benchmark,
)


## Intervention Design

We compare repeated and control prompts, then ablate heads. The key statistic is:

`causal_selective_drop = repeated_ablation_drop - control_ablation_drop`

Positive values mean the ablation hurts repetition-based continuation more than the matched control.


In [ ]:
prompt_family = "synthetic"  # switch to "code" after checking the synthetic baseline
n_prompts = 6 if prompt_family == "synthetic" else 4
seq_len = 8
seed = 0

result, details = run_benchmark(
    prompt_family=prompt_family,
    n_prompts=n_prompts,
    seq_len=seq_len,
    seed=seed,
)
details["result"]


## Candidate Heads

Detection proposes candidates, but the selected head is the top candidate by causal selectivity among the top detected heads. This is where the earlier disagreement between `L1H11` and `L8H6` becomes visible.


In [ ]:
candidate_df = pd.DataFrame(details["top_heads"])
candidate_df.sort_values(["causal_selective_drop", "score"], ascending=False).reset_index(drop=True).head(10)


In [ ]:
detection_layer, detection_head = details["result"]["detection_top_head"]
causal_layer, causal_head = details["result"]["top_head"]

comparison = pd.concat([
    candidate_df.query("layer == @detection_layer and head == @detection_head"),
    candidate_df.query("layer == @causal_layer and head == @causal_head"),
]).drop_duplicates().reset_index(drop=True)
comparison


## Multi-Head Ablations

Later experiments suggested the synthetic effect is stronger under a small cooperating set than under any single head alone. This cell recreates that analysis for `L8H6` and `L4H13`.

In [ ]:
model = load_model(device="cpu")
dataset = build_prompt_dataset(
    model,
    prompt_family=prompt_family,
    n_prompts=n_prompts,
    seq_len=seq_len,
    seed=seed,
)

candidate_head_pool = [(8, 6), (4, 13)]
head_sets = []
for r in range(1, len(candidate_head_pool) + 1):
    head_sets.extend(list(combinations(candidate_head_pool, r)))

head_sets[:3]

In [ ]:
head_set_rows = []
for head_set in head_sets:
    row = evaluate_head_set(model, dataset, list(head_set))
    row["head_set_label"] = " + ".join([f"L{layer}H{head}" for layer, head in head_set])
    head_set_rows.append(row)

head_set_df = pd.DataFrame(head_set_rows)
head_set_df.sort_values(["causal_selective_drop", "repeated_ablation_drop"], ascending=False).reset_index(drop=True)


## Load Modal Artifacts

Heavy sweeps and GPU-backed runs should still happen through Modal. The notebook can then load the saved outputs and keep the interpretation work in one place.


In [ ]:
def load_json_if_present(path: Path):
    if path.exists():
        return json.loads(path.read_text())
    return None

out_dir = ROOT / "outputs" / "induction_heads"
artifact_paths = {
    "synthetic_baseline": out_dir / "modal_latest.json",
    "code_baseline": out_dir / "code_modal_latest.json",
    "synthetic_sweep": out_dir / "sweep_summary.json",
    "synthetic_head_sets": out_dir / "head_sets_synth_n10_l8_s0.json",
    "code_head_sets": out_dir / "head_sets_code_n4.json",
}

artifacts = {name: load_json_if_present(path) for name, path in artifact_paths.items()}
{name: (artifact_paths[name], artifacts[name] is not None) for name in artifact_paths}


In [ ]:
if artifacts["synthetic_baseline"] is not None:
    display(pd.DataFrame([artifacts["synthetic_baseline"]["result"]]))

if artifacts["code_baseline"] is not None:
    display(pd.DataFrame([artifacts["code_baseline"]["result"]]))

if artifacts["synthetic_sweep"] is not None:
    display(pd.DataFrame(artifacts["synthetic_sweep"]))

if artifacts["synthetic_head_sets"] is not None:
    display(pd.DataFrame(artifacts["synthetic_head_sets"]["rows"]))

if artifacts["code_head_sets"] is not None:
    display(pd.DataFrame(artifacts["code_head_sets"]["rows"]))


## Modal Commands

Use the scripts for expensive cloud execution, then come back to this notebook to inspect the saved results:

- `uv run modal run scripts/modal_run.py --prompt-family synthetic --n-prompts 10 --seq-len 8 --seed 0 --save-path outputs/induction_heads/modal_latest.json`
- `uv run modal run scripts/modal_run.py --prompt-family code --n-prompts 4 --seq-len 8 --seed 0 --save-path outputs/induction_heads/code_modal_latest.json`
- `uv run modal run scripts/head_sets_modal.py --prompt-family synthetic --n-prompts 10 --seq-len 8 --seed 0 --save-path outputs/induction_heads/head_sets_synth_n10_l8_s0.json`
- `uv run modal run scripts/head_sets_modal.py --prompt-family code --n-prompts 4 --seq-len 8 --seed 0 --save-path outputs/induction_heads/head_sets_code_n4.json`
